# PyTorch for Regression / Continous Variable

In [7]:
import pandas as pd
import numpy as np
import random
import pyarrow as pa
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.optim as optim
# Import

In [ ]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [2]:
# Load Observation Dataset
# DATA = readRDS(paste0(strap_dir,'Data/Models/RF_bi_model_All_DATA_all_vars_','Trends_Conc_PWS_GW_05to20', '.rds'))
#future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/StRAPs/StRAP4/SSWR.405.1_Future_DW/Data/'
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'


dataset1 = 'Dataset_Conc_Model_SW_COMIDS.csv'
#dataset1 = 'Dataset_Conc_Model_GW_COMIDS.csv'

input_data_ = pd.read_csv(future_dir+dataset1)
input_data_.head(3)

,COMID,mean_conc,PopDen2010Cat,PctCrop2019Cat,precip9120ws,tmean9120cat,BFICat,permcat,RockNCat,N_TW2012Cat,N_Surp_kgsqkm_2017ws,Fe2O3Cat,ElevCat,NHDslope_Pct_Cat
0,12564,0.108571,4.8193,0.00,454.415442,9.029232,65.0000,8.889206,422.7846,3.607679,350.849192,3.6657,1866.1304,15.421640
1,12712,0.780000,42.0676,0.53,660.986854,9.569853,63.9551,7.496483,347.2967,3.707600,648.834773,2.5722,1605.7684,0.819565
2,13230,0.054286,26.6277,0.00,432.323988,9.609029,61.1439,6.617141,385.5680,3.704943,751.514012,3.0452,1716.6487,4.737774


In [5]:
input_data_.columns

Index(['COMID', 'mean_conc', 'PopDen2010Cat', 'PctCrop2019Cat', 'precip9120ws',
       'tmean9120cat', 'BFICat', 'permcat', 'RockNCat', 'N_TW2012Cat',
       'N_Surp_kgsqkm_2017ws', 'Fe2O3Cat', 'ElevCat', 'NHDslope_Pct_Cat'],
      dtype='str')

In [6]:
# Remove extra fields
input_data = input_data_.drop(columns=['COMID']) 
input_data.shape

(1848, 13)

In [24]:
X_input_data = input_data.drop(columns=['mean_conc']) 
type(X_input_data)

pandas.DataFrame

In [8]:
# Convert to tensors and split into train and test sets
from sklearn.model_selection import train_test_split
X = input_data.drop(columns=['mean_conc']).values
y = input_data['mean_conc'].values

# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# Make a copy to use later
X_full = X.clone()
y_full = y.clone()

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2,
                                                    random_state=2
)

C:\Users\MPennino\AppData\Local\Temp\ipykernel_185528\3695896241.py:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at D:\bld\libtorch_1784990676194\work\torch\csrc\utils\tensor_numpy.cpp:219.)
  y = torch.from_numpy(y).type(torch.float)


In [19]:
# Assess Summary of Conc Data
for col in input_data.columns:
    print(f"Mean of {col}: {input_data[col].mean()}")

Mean of mean_conc: 0.6540739088202562
Mean of PopDen2010Cat: 137.04127397186147
Mean of PctCrop2019Cat: 6.087608225108225
Mean of precip9120ws: 1137.8772155973502
Mean of tmean9120cat: 11.011108821186966
Mean of BFICat: 44.8581821969697
Mean of permcat: 7.5577005841076685
Mean of RockNCat: 197.46318755411255
Mean of N_TW2012Cat: 7.626754110778228
Mean of N_Surp_kgsqkm_2017ws: 2224.5156141600633
Mean of Fe2O3Cat: 5.821117532467532
Mean of ElevCat: 579.2841042207793
Mean of NHDslope_Pct_Cat: 1.911423175865801


In [14]:
nrows = X_train.size()[0]
ncols = X_train.size()[1]
nrows,ncols

(1478, 12)

In [34]:


# 1. Generate synthetic non-normal (skewed) continuous data
torch.manual_seed(42)
# X = torch.rand(500, 1) * 10  # Input feature
# # Create a non-normal target (exponential/skewed noise added to a linear relation)
# y = 3.5 * X + 2.0 + torch.exponential_rate(torch.ones(500, 1), lam=0.5) 

# 2. Define a simple Multi-Layer Perceptron (MLP) for non-linear/robust mapping
class RobustRegressionModel(nn.Module):
    def __init__(self, input_dim=ncols, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, x):
        return self.net(x)

model = RobustRegressionModel()

In [46]:
torch.manual_seed(42)

# 2. Define a simple Multi-Layer Perceptron (MLP) for non-linear/robust mapping
class RobustRegressionModel(nn.Module):
    def __init__(self, input_dim=ncols, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.01),             # Prevents dead neurons
            nn.BatchNorm1d(hidden_dim),     # Stabilizes training
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2), 
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, x):
        return self.net(x)

model = RobustRegressionModel()

In [47]:

# 3. Use L1Loss (Mean Absolute Error) to be robust against non-normal/skewed data
criterion = nn.L1Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 4. Training Loop
epochs = 500
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # Forward pass
    predictions = model(X)
    
    # Compute loss
    loss = criterion(predictions, y)
    
    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss (MAE): {loss.item():.4f}")

Epoch [100/500], Loss (MAE): 0.5023
Epoch [200/500], Loss (MAE): 0.5022
Epoch [300/500], Loss (MAE): 0.5021
Epoch [400/500], Loss (MAE): 0.5021
Epoch [500/500], Loss (MAE): 0.5021


# Model Evaluation Metrics

In [42]:
pd.Series(y_test).describe()

count    370.000000
mean       0.568616
std        0.976606
min        0.000000
25%        0.153571
50%        0.257636
75%        0.537857
max        8.818571
dtype: float64

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze()# X_input_data

# Convert to pd dataframe
preds_df = pd.DataFrame(preds.cpu().numpy(), columns=['mean_conc'])
pd.Series(preds_df['mean_conc']).describe()


count    370.000000
mean       0.289937
std        0.001728
min        0.287389
25%        0.289830
50%        0.289904
75%        0.289965
max        0.322384
Name: mean_conc, dtype: float64

In [ ]:
import torch
from torchmetrics.regression import R2Score
# k = 3 independent features/predictors used in the model
k = ncols 

# Instantiate the metric with the 'adjusted' parameter
r2_metric = R2Score(adjusted=k)

# Simulated predictions and ground truths
model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze()# X_input_data
target = y_test

# Calculate Adjusted R²
adj_r2 = r2_metric(preds, target)
print(f"Adjusted R²: {adj_r2.item()}")

RuntimeError: Predictions and targets are expected to have the same shape, but got torch.Size([370, 1]) and torch.Size([370]).

In [31]:
target[0:10], preds[0:10]

(tensor([0.0714, 0.4450, 0.0783, 0.0540, 0.5660, 3.0443, 4.2733, 0.1983, 0.4850,
         0.3900]),
 tensor([0.2900, 0.2900, 0.2900, 0.2900, 0.2900, 0.2725, 0.2918, 0.2900, 0.2900,
         0.2900]))